# Searchlight Institute Tax Reform Analysis - Reform 2

This notebook analyzes the comprehensive impact of the Searchlight Institute tax reform proposal (Reform 2):
- Full aggregate impacts for 2026 (deciles, poverty, winners/losers, income brackets)
- 10-year federal budgetary impact (2026-2035)

**CTC Amounts:** $3,000 (2026) to $3,600 (2035)

In [1]:
from policyengine_us import Microsimulation
from policyengine_core.reforms import Reform
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from policyengine_core.charts import format_fig
import csv

In [2]:
# Searchlight Institute Tax Reform - Reform 2
# - AFA CTC reforms (lower CTC amounts)
# - EITC eligibility and amount changes
# - EITC joint bonus set equal to phase-out start values

REFORM_DICT = {
    "gov.contrib.congress.afa.in_effect": {
        "2026-01-01.2100-12-31": True
    },
    "gov.contrib.congress.afa.ctc.amount.base": {
        "2026-01-01.2026-12-31": 3000,
        "2027-01-01.2028-12-31": 3120,
        "2029-01-01.2030-12-31": 3240,
        "2031-01-01.2032-12-31": 3360,
        "2033-01-01.2033-12-31": 3480,
        "2034-01-01.2035-12-31": 3600
    },
    "gov.irs.credits.eitc.eligibility.age.max": {
        "2026-01-01.2100-12-31": 200
    },
    "gov.irs.credits.eitc.eligibility.age.min": {
        "2026-01-01.2100-12-31": 19
    },
    "gov.irs.credits.eitc.eligibility.age.min_student": {
        "2026-01-01.2100-12-31": 24
    },
    "gov.irs.credits.eitc.max[0].amount": {
        "2026-01-01.2026-12-31": 1502,
        "2027-01-01.2027-12-31": 1577,
        "2028-01-01.2028-12-31": 1610,
        "2029-01-01.2029-12-31": 1645,
        "2030-01-01.2030-12-31": 1678,
        "2031-01-01.2031-12-31": 1712,
        "2032-01-01.2032-12-31": 1746,
        "2033-01-01.2033-12-31": 1779,
        "2034-01-01.2034-12-31": 1815,
        "2035-01-01.2035-12-31": 1851
    },
    "gov.irs.credits.eitc.phase_in_rate[0].amount": {
        "2026-01-01.2100-12-31": 0.153
    },
    "gov.irs.credits.eitc.phase_out.rate[0].amount": {
        "2026-01-01.2100-12-31": 0.153
    },
    "gov.irs.credits.eitc.phase_out.start[0].amount": {
        "2026-01-01.2026-12-31": 11610,
        "2027-01-01.2027-12-31": 12190,
        "2028-01-01.2028-12-31": 12440,
        "2029-01-01.2029-12-31": 12710,
        "2030-01-01.2030-12-31": 12970,
        "2031-01-01.2031-12-31": 13230,
        "2032-01-01.2032-12-31": 13490,
        "2033-01-01.2033-12-31": 13750,
        "2034-01-01.2034-12-31": 14030,
        "2035-01-01.2035-12-31": 14300
    },
    "gov.irs.credits.eitc.phase_out.joint_bonus[0].amount": {
        "2026-01-01.2026-12-31": 11610,
        "2027-01-01.2027-12-31": 12190,
        "2028-01-01.2028-12-31": 12440,
        "2029-01-01.2029-12-31": 12710,
        "2030-01-01.2030-12-31": 12970,
        "2031-01-01.2031-12-31": 13230,
        "2032-01-01.2032-12-31": 13490,
        "2033-01-01.2033-12-31": 13750,
        "2034-01-01.2034-12-31": 14030,
        "2035-01-01.2035-12-31": 14300
    },
    "gov.contrib.congress.afa.ctc.phase_out.threshold.lower.JOINT": {
        "2026-01-01.2026-12-31": 150000,
        "2027-01-01.2028-12-31": 155000,
        "2029-01-01.2029-12-31": 160000,
        "2030-01-01.2031-12-31": 165000,
        "2032-01-01.2033-12-31": 170000,
        "2034-01-01.2034-12-31": 175000,
        "2035-01-01.2035-12-31": 180000
    },
    "gov.contrib.congress.afa.ctc.phase_out.threshold.lower.SINGLE": {
        "2026-01-01.2027-12-31": 112500,
        "2028-01-01.2029-12-31": 117500,
        "2030-01-01.2031-12-31": 122500,
        "2032-01-01.2033-12-31": 127500,
        "2034-01-01.2035-12-31": 132500
    },
    "gov.contrib.congress.afa.ctc.phase_out.threshold.lower.SURVIVING_SPOUSE": {
        "2026-01-01.2026-12-31": 150000,
        "2027-01-01.2028-12-31": 155000,
        "2029-01-01.2029-12-31": 160000,
        "2030-01-01.2031-12-31": 165000,
        "2032-01-01.2033-12-31": 170000,
        "2034-01-01.2034-12-31": 175000,
        "2035-01-01.2035-12-31": 180000
    },
    "gov.contrib.congress.afa.ctc.phase_out.threshold.lower.HEAD_OF_HOUSEHOLD": {
        "2026-01-01.2027-12-31": 112500,
        "2028-01-01.2029-12-31": 117500,
        "2030-01-01.2031-12-31": 122500,
        "2032-01-01.2033-12-31": 127500,
        "2034-01-01.2035-12-31": 132500
    }
}

searchlight_reform = Reform.from_dict(REFORM_DICT, country_id="us")

## Comprehensive 2026 Impact Analysis

In [3]:
# Intra-decile bounds and labels
_INTRA_BOUNDS = [-np.inf, -0.05, -1e-3, 1e-3, 0.05, np.inf]
_INTRA_LABELS = [
    "Lose more than 5%",
    "Lose less than 5%",
    "No change",
    "Gain less than 5%",
    "Gain more than 5%",
]

DATASET = "hf://policyengine/policyengine-us-data/enhanced_cps_2024.h5"


def _poverty_metrics(baseline_rate, reform_rate):
    """Return rate change and percent change for a poverty metric."""
    rate_change = reform_rate - baseline_rate
    percent_change = (
        rate_change / baseline_rate * 100
        if baseline_rate > 0
        else 0.0
    )
    return rate_change, percent_change


def calculate_aggregate_impact(reform, year: int = 2026) -> dict:
    sim_baseline = Microsimulation(dataset=DATASET)
    sim_reform = Microsimulation(reform=reform, dataset=DATASET)

    # ===== FISCAL IMPACT =====
    tax_baseline = sim_baseline.calculate(
        "income_tax", period=year, map_to="household"
    )
    tax_reform = sim_reform.calculate(
        "income_tax", period=year, map_to="household"
    )
    income_change = tax_baseline - tax_reform
    tax_revenue_impact = float(-income_change.sum())

    # Total households
    total_households = float((income_change * 0 + 1).sum())

    # ===== WINNERS / LOSERS =====
    winners = float((income_change > 1).sum())
    losers = float((income_change < -1).sum())
    beneficiaries = float((income_change > 0).sum())

    affected = abs(income_change) > 1
    affected_count = float(affected.sum())

    household_weight_early = sim_reform.calculate(
        "household_weight", period=year
    )
    affected_mask = np.array(affected).astype(bool)
    change_arr_early = np.array(income_change)
    weight_arr_early = np.array(household_weight_early)
    avg_benefit = (
        float(np.average(
            change_arr_early[affected_mask],
            weights=weight_arr_early[affected_mask],
        ))
        if affected_count > 0
        else 0.0
    )

    winners_rate = winners / total_households * 100
    losers_rate = losers / total_households * 100

    # ===== INCOME DECILE ANALYSIS =====
    decile = sim_baseline.calculate(
        "household_income_decile", period=year, map_to="household"
    )
    baseline_net_income = sim_baseline.calculate(
        "household_net_income", period=year, map_to="household"
    )

    decile_average = {}
    decile_relative = {}
    for d in range(1, 11):
        dmask = decile == d
        d_count = float(dmask.sum())
        if d_count > 0:
            d_change_sum = float(income_change[dmask].sum())
            decile_average[str(d)] = d_change_sum / d_count
            d_baseline_sum = float(baseline_net_income[dmask].sum())
            decile_relative[str(d)] = (
                d_change_sum / d_baseline_sum
                if d_baseline_sum != 0
                else 0.0
            )
        else:
            decile_average[str(d)] = 0.0
            decile_relative[str(d)] = 0.0

    # Intra-decile
    household_weight = sim_reform.calculate(
        "household_weight", period=year
    )
    people_per_hh = sim_baseline.calculate(
        "household_count_people", period=year, map_to="household"
    )
    capped_baseline = np.maximum(np.array(baseline_net_income), 1)
    rel_change_arr = np.array(income_change) / capped_baseline

    decile_arr = np.array(decile)
    weight_arr = np.array(household_weight)
    people_weighted = np.array(people_per_hh) * weight_arr

    intra_decile_deciles = {label: [] for label in _INTRA_LABELS}
    for d in range(1, 11):
        dmask = decile_arr == d
        d_people = people_weighted[dmask]
        d_total_people = d_people.sum()
        d_rel = rel_change_arr[dmask]

        for lower, upper, label in zip(
            _INTRA_BOUNDS[:-1], _INTRA_BOUNDS[1:], _INTRA_LABELS
        ):
            in_group = (d_rel > lower) & (d_rel <= upper)
            proportion = (
                float(d_people[in_group].sum() / d_total_people)
                if d_total_people > 0
                else 0.0
            )
            intra_decile_deciles[label].append(proportion)

    intra_decile_all = {
        label: sum(intra_decile_deciles[label]) / 10
        for label in _INTRA_LABELS
    }

    # ===== POVERTY IMPACT =====
    pov_bl = sim_baseline.calculate(
        "in_poverty", period=year, map_to="person"
    )
    pov_rf = sim_reform.calculate(
        "in_poverty", period=year, map_to="person"
    )
    poverty_baseline_rate = float(pov_bl.mean() * 100)
    poverty_reform_rate = float(pov_rf.mean() * 100)
    poverty_rate_change, poverty_percent_change = _poverty_metrics(
        poverty_baseline_rate, poverty_reform_rate
    )

    # Child/deep poverty
    age_arr = np.array(sim_baseline.calculate("age", period=year))
    is_child = age_arr < 18
    pw_arr = np.array(sim_baseline.calculate("person_weight", period=year))
    child_w = pw_arr[is_child]
    total_child_w = child_w.sum()

    pov_bl_arr = np.array(pov_bl).astype(bool)
    pov_rf_arr = np.array(pov_rf).astype(bool)

    def _child_rate(arr):
        return float(
            (arr[is_child] * child_w).sum() / total_child_w * 100
        ) if total_child_w > 0 else 0.0

    child_poverty_baseline_rate = _child_rate(pov_bl_arr)
    child_poverty_reform_rate = _child_rate(pov_rf_arr)
    child_poverty_rate_change, child_poverty_percent_change = (
        _poverty_metrics(
            child_poverty_baseline_rate, child_poverty_reform_rate
        )
    )

    deep_bl = sim_baseline.calculate(
        "in_deep_poverty", period=year, map_to="person"
    )
    deep_rf = sim_reform.calculate(
        "in_deep_poverty", period=year, map_to="person"
    )
    deep_poverty_baseline_rate = float(deep_bl.mean() * 100)
    deep_poverty_reform_rate = float(deep_rf.mean() * 100)
    deep_poverty_rate_change, deep_poverty_percent_change = (
        _poverty_metrics(
            deep_poverty_baseline_rate, deep_poverty_reform_rate
        )
    )

    deep_bl_arr = np.array(deep_bl).astype(bool)
    deep_rf_arr = np.array(deep_rf).astype(bool)
    deep_child_poverty_baseline_rate = _child_rate(deep_bl_arr)
    deep_child_poverty_reform_rate = _child_rate(deep_rf_arr)
    deep_child_poverty_rate_change, deep_child_poverty_percent_change = (
        _poverty_metrics(
            deep_child_poverty_baseline_rate,
            deep_child_poverty_reform_rate,
        )
    )

    # ===== INCOME BRACKET BREAKDOWN =====
    agi = sim_reform.calculate(
        "adjusted_gross_income", period=year, map_to="household"
    )
    agi_arr = np.array(agi)
    change_arr = np.array(income_change)
    affected_mask = np.abs(change_arr) > 1

    income_brackets = [
        (0, 50_000, "Under $50k"),
        (50_000, 100_000, "$50k-$100k"),
        (100_000, 200_000, "$100k-$200k"),
        (200_000, 500_000, "$200k-$500k"),
        (500_000, 1_000_000, "$500k-$1M"),
        (1_000_000, 2_000_000, "$1M-$2M"),
        (2_000_000, float("inf"), "Over $2M"),
    ]

    by_income_bracket = []
    for min_inc, max_inc, label in income_brackets:
        mask = (
            (agi_arr >= min_inc)
            & (agi_arr < max_inc)
            & affected_mask
        )
        bracket_affected = float(weight_arr[mask].sum())
        if bracket_affected > 0:
            bracket_cost = float(
                (change_arr[mask] * weight_arr[mask]).sum()
            )
            bracket_avg = float(
                np.average(change_arr[mask], weights=weight_arr[mask])
            )
        else:
            bracket_cost = 0.0
            bracket_avg = 0.0
        by_income_bracket.append({
            "bracket": label,
            "beneficiaries": bracket_affected,
            "total_cost": bracket_cost,
            "avg_benefit": bracket_avg,
        })

    return {
        "budget": {
            "budgetary_impact": tax_revenue_impact,
            "tax_revenue_impact": tax_revenue_impact,
            "benefit_spending_impact": 0.0,
            "households": total_households,
        },
        "decile": {
            "average": decile_average,
            "relative": decile_relative,
        },
        "intra_decile": {
            "all": intra_decile_all,
            "deciles": intra_decile_deciles,
        },
        "total_cost": -tax_revenue_impact,
        "beneficiaries": beneficiaries,
        "avg_benefit": avg_benefit,
        "winners": winners,
        "losers": losers,
        "winners_rate": winners_rate,
        "losers_rate": losers_rate,
        "poverty_baseline_rate": poverty_baseline_rate,
        "poverty_reform_rate": poverty_reform_rate,
        "poverty_rate_change": poverty_rate_change,
        "poverty_percent_change": poverty_percent_change,
        "child_poverty_baseline_rate": child_poverty_baseline_rate,
        "child_poverty_reform_rate": child_poverty_reform_rate,
        "child_poverty_rate_change": child_poverty_rate_change,
        "child_poverty_percent_change": child_poverty_percent_change,
        "deep_poverty_baseline_rate": deep_poverty_baseline_rate,
        "deep_poverty_reform_rate": deep_poverty_reform_rate,
        "deep_poverty_rate_change": deep_poverty_rate_change,
        "deep_poverty_percent_change": deep_poverty_percent_change,
        "deep_child_poverty_baseline_rate": deep_child_poverty_baseline_rate,
        "deep_child_poverty_reform_rate": deep_child_poverty_reform_rate,
        "deep_child_poverty_rate_change": deep_child_poverty_rate_change,
        "deep_child_poverty_percent_change": deep_child_poverty_percent_change,
        "by_income_bracket": by_income_bracket,
    }

In [4]:
# Calculate comprehensive 2026 impacts
print("Calculating comprehensive 2026 impacts...")
impact_2026 = calculate_aggregate_impact(searchlight_reform, year=2026)
print("Done!")

# Export 2026 impact to CSV immediately
def export_2026_impact_to_csv(impact, filename):
    """Export the 2026 impact results to a CSV file."""
    rows = [
        ["Metric", "Value"],
        ["Budgetary Impact ($)", impact['budget']['budgetary_impact']],
        ["Total Cost ($)", impact['total_cost']],
        ["Total Households", impact['budget']['households']],
        ["Winners", impact['winners']],
        ["Winners Rate (%)", impact['winners_rate']],
        ["Losers", impact['losers']],
        ["Losers Rate (%)", impact['losers_rate']],
        ["Beneficiaries", impact['beneficiaries']],
        ["Average Benefit ($)", impact['avg_benefit']],
        ["Poverty Baseline Rate (%)", impact['poverty_baseline_rate']],
        ["Poverty Reform Rate (%)", impact['poverty_reform_rate']],
        ["Poverty Rate Change (pp)", impact['poverty_rate_change']],
        ["Poverty Percent Change (%)", impact['poverty_percent_change']],
        ["Child Poverty Baseline Rate (%)", impact['child_poverty_baseline_rate']],
        ["Child Poverty Reform Rate (%)", impact['child_poverty_reform_rate']],
        ["Child Poverty Rate Change (pp)", impact['child_poverty_rate_change']],
        ["Child Poverty Percent Change (%)", impact['child_poverty_percent_change']],
        ["Deep Poverty Baseline Rate (%)", impact['deep_poverty_baseline_rate']],
        ["Deep Poverty Reform Rate (%)", impact['deep_poverty_reform_rate']],
        ["Deep Poverty Rate Change (pp)", impact['deep_poverty_rate_change']],
        ["Deep Poverty Percent Change (%)", impact['deep_poverty_percent_change']],
        ["Deep Child Poverty Baseline Rate (%)", impact['deep_child_poverty_baseline_rate']],
        ["Deep Child Poverty Reform Rate (%)", impact['deep_child_poverty_reform_rate']],
        ["Deep Child Poverty Rate Change (pp)", impact['deep_child_poverty_rate_change']],
        ["Deep Child Poverty Percent Change (%)", impact['deep_child_poverty_percent_change']],
    ]
    
    # Add decile data
    for d in range(1, 11):
        rows.append([f"Decile {d} Average Change ($)", impact['decile']['average'][str(d)]])
        rows.append([f"Decile {d} Relative Change", impact['decile']['relative'][str(d)]])
    
    # Add income bracket data
    for bracket in impact['by_income_bracket']:
        rows.append([f"Bracket {bracket['bracket']} Beneficiaries", bracket['beneficiaries']])
        rows.append([f"Bracket {bracket['bracket']} Total Cost ($)", bracket['total_cost']])
        rows.append([f"Bracket {bracket['bracket']} Avg Benefit ($)", bracket['avg_benefit']])
    
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerows(rows)
    
    print(f"2026 impact saved to {filename}")

export_2026_impact_to_csv(impact_2026, "reform_2_impact_2026.csv")

Calculating comprehensive 2026 impacts...
Done!
2026 impact saved to reform_2_impact_2026.csv


In [5]:
# Display budget summary
print("=" * 60)
print("BUDGET SUMMARY (2026) - Reform 2")
print("=" * 60)
print(f"Budgetary Impact: ${impact_2026['budget']['budgetary_impact'] / 1e9:.2f} billion")
print(f"Total Cost: ${impact_2026['total_cost'] / 1e9:.2f} billion")
print(f"Total Households: {impact_2026['budget']['households']:,.0f}")

BUDGET SUMMARY (2026) - Reform 2
Budgetary Impact: $-113.44 billion
Total Cost: $113.44 billion
Total Households: 147,829,631


In [6]:
# Display winners/losers
print("\n" + "=" * 60)
print("WINNERS / LOSERS (2026)")
print("=" * 60)
print(f"Winners: {impact_2026['winners']:,.0f} ({impact_2026['winners_rate']:.1f}%)")
print(f"Losers: {impact_2026['losers']:,.0f} ({impact_2026['losers_rate']:.1f}%)")
print(f"Beneficiaries: {impact_2026['beneficiaries']:,.0f}")
print(f"Average Benefit (affected): ${impact_2026['avg_benefit']:,.2f}")


WINNERS / LOSERS (2026)
Winners: 46,577,942 (31.5%)
Losers: 5,164,542 (3.5%)
Beneficiaries: 46,580,372
Average Benefit (affected): $2,192.45


In [7]:
# Display poverty impacts
print("\n" + "=" * 60)
print("POVERTY IMPACTS (2026)")
print("=" * 60)
print(f"\nOverall Poverty:")
print(f"  Baseline: {impact_2026['poverty_baseline_rate']:.2f}%")
print(f"  Reform: {impact_2026['poverty_reform_rate']:.2f}%")
print(f"  Change: {impact_2026['poverty_rate_change']:+.2f}pp ({impact_2026['poverty_percent_change']:+.1f}%)")

print(f"\nChild Poverty:")
print(f"  Baseline: {impact_2026['child_poverty_baseline_rate']:.2f}%")
print(f"  Reform: {impact_2026['child_poverty_reform_rate']:.2f}%")
print(f"  Change: {impact_2026['child_poverty_rate_change']:+.2f}pp ({impact_2026['child_poverty_percent_change']:+.1f}%)")

print(f"\nDeep Poverty:")
print(f"  Baseline: {impact_2026['deep_poverty_baseline_rate']:.2f}%")
print(f"  Reform: {impact_2026['deep_poverty_reform_rate']:.2f}%")
print(f"  Change: {impact_2026['deep_poverty_rate_change']:+.2f}pp ({impact_2026['deep_poverty_percent_change']:+.1f}%)")

print(f"\nDeep Child Poverty:")
print(f"  Baseline: {impact_2026['deep_child_poverty_baseline_rate']:.2f}%")
print(f"  Reform: {impact_2026['deep_child_poverty_reform_rate']:.2f}%")
print(f"  Change: {impact_2026['deep_child_poverty_rate_change']:+.2f}pp ({impact_2026['deep_child_poverty_percent_change']:+.1f}%)")


POVERTY IMPACTS (2026)

Overall Poverty:
  Baseline: 23.33%
  Reform: 21.58%
  Change: -1.76pp (-7.5%)

Child Poverty:
  Baseline: 22.00%
  Reform: 17.24%
  Change: -4.76pp (-21.6%)

Deep Poverty:
  Baseline: 8.70%
  Reform: 7.46%
  Change: -1.24pp (-14.3%)

Deep Child Poverty:
  Baseline: 8.56%
  Reform: 5.57%
  Change: -2.99pp (-35.0%)


In [8]:
# Display decile analysis
print("\n" + "=" * 60)
print("DECILE ANALYSIS (2026)")
print("=" * 60)
print(f"{'Decile':<10} {'Avg Change':>15} {'Relative Change':>18}")
print("-" * 45)
for d in range(1, 11):
    avg = impact_2026['decile']['average'][str(d)]
    rel = impact_2026['decile']['relative'][str(d)] * 100
    print(f"{d:<10} ${avg:>14,.0f} {rel:>17.2f}%")


DECILE ANALYSIS (2026)
Decile          Avg Change    Relative Change
---------------------------------------------
1          $           781              6.78%
2          $           820              3.06%
3          $           716              1.78%
4          $         1,091              2.01%
5          $         1,369              1.94%
6          $           839              0.97%
7          $           785              0.71%
8          $           724              0.50%
9          $           287              0.15%
10         $           128              0.02%


In [9]:
# Display income bracket breakdown
print("\n" + "=" * 60)
print("INCOME BRACKET BREAKDOWN (2026)")
print("=" * 60)
print(f"{'Bracket':<15} {'Beneficiaries':>15} {'Total Cost':>15} {'Avg Benefit':>12}")
print("-" * 60)
for bracket in impact_2026['by_income_bracket']:
    print(f"{bracket['bracket']:<15} {bracket['beneficiaries']:>15,.0f} ${bracket['total_cost']/1e9:>13.2f}B ${bracket['avg_benefit']:>10,.0f}")


INCOME BRACKET BREAKDOWN (2026)
Bracket           Beneficiaries      Total Cost  Avg Benefit
------------------------------------------------------------
Under $50k           21,366,040 $        59.49B $     2,784
$50k-$100k           12,473,454 $        29.25B $     2,345
$100k-$200k          11,142,482 $        20.05B $     1,799
$200k-$500k           5,414,826 $         0.93B $       171
$500k-$1M               335,220 $         0.57B $     1,707
$1M-$2M                  31,084 $         0.03B $     1,045
Over $2M                  8,306 $         0.01B $     1,042


In [10]:
# Create decile bar chart
deciles = list(range(1, 11))
avg_changes = [impact_2026['decile']['average'][str(d)] for d in deciles]

fig = go.Figure(
    go.Bar(
        x=deciles,
        y=avg_changes,
        text=[f"${v:,.0f}" for v in avg_changes],
        textposition="auto",
        marker_color="#105293",
    )
)

fig.update_layout(
    title="Average Income Change by Decile (2026) - Reform 2",
    xaxis_title="Income Decile",
    yaxis_title="Average Change ($)",
    xaxis=dict(tickmode="linear"),
)

format_fig(fig).show()

## 10-Year Federal Budgetary Impact (2026-2035)

In [11]:
def calculate_budgetary_impact(reform, year):
    """Calculate just the federal income tax budgetary impact for a single year."""
    baseline = Microsimulation(dataset=DATASET)
    reformed = Microsimulation(reform=reform, dataset=DATASET)
    
    baseline_income_tax = baseline.calculate("income_tax", period=year).sum()
    reformed_income_tax = reformed.calculate("income_tax", period=year).sum()
    
    # Budgetary impact: negative means costs money (reform reduces tax revenue)
    impact = reformed_income_tax - baseline_income_tax
    return float(impact)


def calculate_ten_year_projection(reform, csv_filename="reform_2_10year_budgetary_impact.csv"):
    results = []

    with open(csv_filename, "w", newline="") as csvfile:
        csvwriter = csv.writer(csvfile)
        csvwriter.writerow(["Year", "Budgetary Impact"])

        for year in range(2026, 2036):
            print(f"Computing budgetary impact for year {year}...")
            impact = calculate_budgetary_impact(reform, year)
            results.append({"Year": year, "Budgetary Impact": impact})

            csvwriter.writerow([year, impact])
            print(f"Year {year} completed. Impact: ${impact/1e9:.2f} billion")

    print(f"All calculations complete. Results saved to '{csv_filename}'")
    return pd.DataFrame(results)

In [12]:
# Calculate 10-year projection
print("Starting 10-year budgetary impact calculation...")
df_projection = calculate_ten_year_projection(searchlight_reform)
print("\nCalculation complete. Summary of results:")
print(df_projection)
print(f"\n10-Year Total: ${df_projection['Budgetary Impact'].sum() / 1e9:.2f} billion")

Starting 10-year budgetary impact calculation...
Computing budgetary impact for year 2026...
Year 2026 completed. Impact: $-113.44 billion
Computing budgetary impact for year 2027...
Year 2027 completed. Impact: $-116.92 billion
Computing budgetary impact for year 2028...
Year 2028 completed. Impact: $-115.48 billion
Computing budgetary impact for year 2029...
Year 2029 completed. Impact: $-118.02 billion
Computing budgetary impact for year 2030...
Year 2030 completed. Impact: $-117.95 billion
Computing budgetary impact for year 2031...
Year 2031 completed. Impact: $-120.34 billion
Computing budgetary impact for year 2032...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


enhanced_cps_2024.h5:   0%|          | 0.00/111M [00:00<?, ?B/s]

Year 2032 completed. Impact: $-119.07 billion
Computing budgetary impact for year 2033...
Year 2033 completed. Impact: $-121.04 billion
Computing budgetary impact for year 2034...
Year 2034 completed. Impact: $-127.73 billion
Computing budgetary impact for year 2035...
Year 2035 completed. Impact: $-122.59 billion
All calculations complete. Results saved to 'reform_2_10year_budgetary_impact.csv'

Calculation complete. Summary of results:
   Year  Budgetary Impact
0  2026     -1.134431e+11
1  2027     -1.169174e+11
2  2028     -1.154835e+11
3  2029     -1.180234e+11
4  2030     -1.179546e+11
5  2031     -1.203397e+11
6  2032     -1.190688e+11
7  2033     -1.210389e+11
8  2034     -1.277271e+11
9  2035     -1.225942e+11

10-Year Total: $-1192.59 billion


In [ ]:
# Create 10-year bar chart
fig = go.Figure(
    go.Bar(
        x=df_projection["Year"],
        y=(df_projection["Budgetary Impact"] / 1e9).round(1),
        text=(df_projection["Budgetary Impact"] / 1e9)
        .round(1)
        .apply(lambda x: f"${x:.1f}B"),
        textposition="auto",
        marker_color="#105293",
    )
)

fig.update_layout(
    title="10-Year Federal Budgetary Impact - Reform 2",
    xaxis_title="Year",
    yaxis_title="Budgetary Impact (Billions $)",
    xaxis=dict(tickmode="linear"),
    yaxis=dict(zeroline=True, zerolinewidth=2, zerolinecolor="Black"),
)

format_fig(fig).show()

: 